# Blaban Sales Data Analysis

## Business Performance, Demand & Operations Analysis

This project analyzes Blaban sales data to identify key business insights related to sales performance, product demand, customer behavior, discounts, branch and regional performance, and delivery operations.


In [ ]:
import pandas as pd

df = pd.read_csv('/content/blaban_cleaned_data 5')

print("Dataset shape:", df.shape)

Dataset shape: (6999, 30)


In [ ]:
# Convert Date_Time to datetime
df['Date_Time'] = pd.to_datetime(df['Date_Time'])

print("Date_Time dtype:", df['Date_Time'].dtype)
print("Start date:", df['Date_Time'].min())
print("End date:", df['Date_Time'].max())

Date_Time dtype: datetime64[ns]
Start date: 2024-01-01 00:00:00
End date: 2024-04-14 03:45:00


## 1. Data Validation

In [ ]:
print("Missing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

Missing values:
Transaction_ID            0
Date_Time                 0
Branch                    0
Product_Name              0
Size                      0
Unit_Price                0
Quantity                  0
Discount_Rate             0
Topping_Type              0
Customer_ID               0
Customer_Age              0
Customer_Gender           0
Membership_Status         0
Payment_Method            0
Order_Source              0
Delivery_Time_Min         0
Delivery_Distance_KM      0
Staff_ID                  0
Temperature_Celsius       0
Humidity_Percent          0
Store_Rating            831
Is_Public_Holiday         0
Category                  0
Region                    0
Is_Weekend                0
Hour_of_Day               0
Tax_Amount                0
Total_Sales               0
Net_price                 0
Month_name                0
dtype: int64

Data types:
Transaction_ID                   int64
Date_Time               datetime64[ns]
Branch                          object
P

In [ ]:
print("Total rows:", len(df))
print("Unique transactions:", df['Transaction_ID'].nunique())
print("Duplicate Transaction_IDs:", df['Transaction_ID'].duplicated().sum())

Total rows: 6999
Unique transactions: 6999
Duplicate Transaction_IDs: 0


In [ ]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Transaction_ID,6999.0,104962.095299,100001.0,102476.0,104926.0,107465.5,110000.0,2883.002213
Date_Time,6999,2024-02-21 16:16:25.769395456,2024-01-01 00:00:00,2024-01-26 18:45:00,2024-02-21 07:15:00,2024-03-18 18:07:30,2024-04-14 03:45:00,NaN
Unit_Price,6999.0,109.904881,40.01,75.175,110.16,144.205,179.96,40.284392
Quantity,6999.0,6.026289,1.0,3.0,6.0,9.0,11.0,3.141858
Discount_Rate,6999.0,0.030319,0.0,0.0,0.0,0.05,0.15,0.051314
Customer_ID,6999.0,70077.804829,50001.0,60134.5,70101.0,80055.5,89988.0,11462.544127
Customer_Age,6999.0,43.276897,12.0,29.0,44.0,57.0,74.0,17.090982
Delivery_Time_Min,6999.0,64.708673,10.0,37.0,65.0,91.0,119.0,31.545417
Delivery_Distance_KM,6999.0,7.705058,0.5,4.1,7.7,11.3,15.0,4.176184
Staff_ID,6999.0,124.5988,101.0,112.5,124.0,137.0,149.0,14.064581


The dataset contains 6,999 unique transactions. No duplicate transaction IDs were found, and the numerical variables fall within reasonable ranges.
Store_Rating contains 831 missing values, so rating-based analyses use available non-null ratings.
Date_Time was converted to datetime format to ensure correct date and time handling during analysis.
The dataset covers the period from January 1, 2024 to April 14, 2024. Therefore, April represents a partial month and should not be used for full-month seasonal comparisons.

## 2. Business Overview

In [ ]:
total_sales = df['Total_Sales'].sum()
total_orders = df['Transaction_ID'].nunique()
aov = df['Total_Sales'].mean()
total_quantity = df['Quantity'].sum()
avg_delivery = df['Delivery_Time_Min'].mean()
avg_rating = df['Store_Rating'].mean()

print("Total Sales:", round(total_sales, 2))
print("Total Orders:", total_orders)
print("Average Order Value:", round(aov, 2))
print("Total Quantity:", total_quantity)
print("Average Delivery Time:", round(avg_delivery, 2))
print("Average Store Rating:", round(avg_rating, 2))

Total Sales: 5131945.58
Total Orders: 6999
Average Order Value: 733.24
Total Quantity: 42178
Average Delivery Time: 64.71
Average Store Rating: 2.98


## 3. Sales Performance

### Sales by Category

In [ ]:
category_analysis = df.groupby('Category').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean'),
    Avg_Quantity=('Quantity', 'mean')
).sort_values('Total_Sales', ascending=False)

category_analysis

,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Category,,,,
Traditional,2563,1912969.00,746.378853,6.060086
Modern,1810,1318041.16,728.199536,6.043094
Signature,1776,1295571.28,729.488333,6.026464
Ice Cream,850,605364.14,712.193106,5.888235


**Insight:** Traditional generates the highest total sales, mainly driven by its higher order volume, while average quantity per order remains similar across categories.

### Product Performance & Best Sellers

In [ ]:
product_analysis = df.groupby('Product_Name').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean'),
    Avg_Quantity=('Quantity', 'mean')
).sort_values('Total_Sales', ascending=False)

product_analysis

,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Product_Name,,,,
Sweet Koshary Mix,932,684076.82,733.988004,6.059013
Qishtouza Nutella,936,681474.28,728.070812,6.141026
Rice Pudding Oven,887,652495.89,735.621071,5.997745
Om Ali Cream,855,642342.05,751.277251,6.057310
Qishtouza Lotus,874,636566.88,728.337391,5.938215
Rice Pudding Nuts,821,618131.06,752.900195,6.130329
Super B.Laban,844,611494.46,724.519502,5.990521
Ice Cream Mango,850,605364.14,712.193106,5.888235


In [ ]:
top_products = product_analysis.sort_values(
    'Total_Sales', ascending=False
).head(5)

bottom_products = product_analysis.sort_values(
    'Total_Sales', ascending=True
).head(5)

print("Top 5 Products by Sales:")
display(top_products)

print("\nBottom 5 Products by Sales:")
display(bottom_products)

Top 5 Products by Sales:


,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Product_Name,,,,
Sweet Koshary Mix,932,684076.82,733.988004,6.059013
Qishtouza Nutella,936,681474.28,728.070812,6.141026
Rice Pudding Oven,887,652495.89,735.621071,5.997745
Om Ali Cream,855,642342.05,751.277251,6.057310
Qishtouza Lotus,874,636566.88,728.337391,5.938215



Bottom 5 Products by Sales:


,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Product_Name,,,,
Ice Cream Mango,850,605364.14,712.193106,5.888235
Super B.Laban,844,611494.46,724.519502,5.990521
Rice Pudding Nuts,821,618131.06,752.900195,6.130329
Qishtouza Lotus,874,636566.88,728.337391,5.938215
Om Ali Cream,855,642342.05,751.277251,6.057310


**Insight:**  
Qishtouza Nutella has the highest order volume, while Sweet Koshary Mix generates the highest total sales. Product sales are largely aligned with order volume, with relatively similar average quantities across products.

### Sales by Branch

In [ ]:
branch_analysis = df.groupby('Branch').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean')
).sort_values('Total_Sales', ascending=False)

branch_analysis

,Orders,Total_Sales,Avg_Order_Value
Branch,,,
Cairo-Nasr City,898,668905.22,744.883318
Cairo-Zamalek,853,657230.39,770.492837
Giza-Zayed,896,654478.62,730.444888
Riyadh-Olaya,879,644244.57,732.928976
Dubai-Marina,869,640900.08,737.514476
Mansoura-Mashaya,885,629287.80,711.059661
Tanta-Saeed,872,625966.35,717.851319
Alex-Stanly,847,610932.55,721.289906


**Insight:** Cairo-Nasr City leads in total sales, driven by the highest order volume, while Cairo-Zamalek has the highest average order value.

### Sales by Region

In [ ]:
region_analysis = df.groupby('Region').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean'),
    Avg_Quantity=('Quantity', 'mean')
).sort_values('Total_Sales', ascending=False)

region_analysis

,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Region,,,,
Greater Cairo,2647,1980614.23,748.248670,6.127314
Delta,1757,1255254.15,714.430364,5.956744
KSA,879,644244.57,732.928976,5.944255
UAE,869,640900.08,737.514476,6.063291
Alexandria,847,610932.55,721.289906,5.902007


**Insight:**  
Greater Cairo generates the highest total sales and has the highest average order value, while Delta has a noticeably lower average order value.

### Sales by Topping Type

In [ ]:
topping_product = pd.crosstab(
    df['Product_Name'],
    df['Topping_Type'],
    normalize='index'
)

topping_product

Topping_Type,Lotus,No Topping,Nutella,Nuts,Pistachio
Product_Name,,,,,
Ice Cream Mango,0.165882,0.267059,0.171765,0.194118,0.201176
Om Ali Cream,0.170760,0.319298,0.148538,0.185965,0.175439
Qishtouza Lotus,0.184211,0.275744,0.194508,0.189931,0.155606
Qishtouza Nutella,0.175214,0.290598,0.186966,0.189103,0.158120
Rice Pudding Nuts,0.164434,0.293544,0.203410,0.169306,0.169306
Rice Pudding Oven,0.196167,0.270575,0.189402,0.170237,0.173619
Super B.Laban,0.206161,0.285545,0.154028,0.189573,0.164692
Sweet Koshary Mix,0.183476,0.280043,0.170601,0.187768,0.178112


In [ ]:
topping_analysis = df.groupby('Topping_Type').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean'),
    Avg_Quantity=('Quantity', 'mean')
).sort_values('Total_Sales', ascending=False)

topping_analysis

,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Topping_Type,,,,
No Topping,1996,1484208.39,743.591378,6.130762
Nuts,1292,919914.68,712.008266,5.944272
Nutella,1242,915233.56,736.903027,6.056361
Pistachio,1203,909647.78,756.149443,6.033250
Lotus,1266,902941.17,713.223673,5.909163


Pistachio orders have the highest average order value, while Nuts and Lotus orders have the lowest average order values

### Sales by Order Source

In [ ]:
source_analysis = df.groupby('Order_Source').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean')
).sort_values('Total_Sales', ascending=False)

source_analysis

,Orders,Total_Sales,Avg_Order_Value
Order_Source,,,
Talabat,1803,1324640.91,734.687138
ElMenus,1778,1275492.49,717.374854
Mobile App,1716,1266389.36,737.989138
In-store,1702,1265422.82,743.491669


**Finding:**  
Talabat generates the highest total sales mainly due to its higher order volume, while In-store orders have the highest average order value.

### Sales by Payment Method

In [ ]:
payment_analysis = df.groupby('Payment_Method').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Order_Value=('Total_Sales', 'mean'),
    Avg_Quantity=('Quantity', 'mean')
).sort_values('Total_Sales', ascending=False)

payment_analysis

,Orders,Total_Sales,Avg_Order_Value,Avg_Quantity
Payment_Method,,,,
Credit Card,1752,1304869.42,744.788482,6.049087
Cash,1744,1303641.30,747.500745,6.103784
Wallet,1744,1266614.33,726.269685,5.953555
Instapay,1759,1256820.53,714.508545,5.998863


### Monthly Sales Overview

Monthly performance was reviewed for the complete months of January, February, and March. April was excluded because the dataset only covers part of the month.

In [ ]:
monthly_analysis = (
    df[df['Month_name'] != 'April']
    .groupby('Month_name')
    .agg(
        Orders=('Transaction_ID', 'nunique'),
        Total_Sales=('Total_Sales', 'sum'),
        Avg_Order_Value=('Total_Sales', 'mean')
    )
    .reindex(['January', 'February', 'March'])
)

monthly_analysis

,Orders,Total_Sales,Avg_Order_Value
Month_name,,,
January,2101,1583952.14,753.903922
February,1985,1477420.38,744.292383
March,2048,1467898.66,716.747393


### Discount Performance

In [ ]:
discount_analysis = df.groupby('Discount_Rate').agg(
    Orders=('Transaction_ID', 'nunique'),
    Total_Sales=('Total_Sales', 'sum'),
    Avg_Sales_per_Order=('Total_Sales', 'mean')
).sort_index()

discount_analysis

,Orders,Total_Sales,Avg_Sales_per_Order
Discount_Rate,,,
0.00,4888,3681663.69,753.204519
0.05,695,511799.11,736.401597
0.10,699,481343.89,688.617868
0.15,717,457138.89,637.571674


In [ ]:
discount_quantity = df.groupby('Discount_Rate').agg(
    Orders=('Transaction_ID', 'nunique'),
    Avg_Quantity=('Quantity', 'mean'),
    Avg_Net_Price=('Net_price', 'mean')
).sort_index()

discount_quantity

,Orders,Avg_Quantity,Avg_Net_Price
Discount_Rate,,,
0.00,4888,5.996318,660.705661
0.05,695,6.260432,609.726691
0.10,699,6.165951,536.327425
0.15,717,5.867503,465.298312


**Finding:**  
Average sales per order are lower at higher discount levels, while average quantity per order does not show a consistent increase.

## 4. Customer Analysis

Customer Overview

In [ ]:
customer_analysis = {
    'Total_Customers': df['Customer_ID'].nunique(),
    'Avg_Spend_per_Customer': df.groupby('Customer_ID')['Total_Sales'].sum().mean(),
    'Avg_Items_per_Order': df['Quantity'].mean()
}

customer_analysis

{'Total_Customers': 6415,
 'Avg_Spend_per_Customer': np.float64(799.9915167575994),
 'Avg_Items_per_Order': np.float64(6.026289469924275)}

Membership Performance

In [ ]:
membership_analysis = df.groupby('Membership_Status').agg(
    Customers=('Customer_ID', 'nunique'),
    Orders=('Transaction_ID', 'nunique'),
    Avg_Quantity=('Quantity', 'mean'),
    Avg_Order_Value=('Total_Sales', 'mean'),
    Total_Sales=('Total_Sales', 'sum')
).sort_values('Total_Sales', ascending=False)

membership_analysis

,Customers,Orders,Avg_Quantity,Avg_Order_Value,Total_Sales
Membership_Status,,,,,
No Membership,2266,2335,5.957602,729.175589,1702625.00
Gold,1568,1601,5.986883,723.667320,1158591.38
Silver,1525,1556,6.129177,738.391510,1148937.19
Bronze,1484,1507,6.068348,744.387532,1121792.01


**Finding:**  
Membership groups show some differences in average order value and quantity, but no strong pattern is large enough to represent a major business insight.

In [ ]:
customer_orders = df.groupby('Customer_ID')['Transaction_ID'].nunique()

returning_customers = (customer_orders > 1).sum()

print("Total Customers:", df['Customer_ID'].nunique())
print("Returning Customers:", returning_customers)

Total Customers: 6415
Returning Customers: 557


**Returning Customer:** A customer who placed more than one order during the analyzed period.

## 5. Operations & Customer Experience

### Delivery Performance

In [ ]:
delivery_summary = df['Delivery_Time_Min'].agg(
    ['count', 'mean', 'min', 'max']
)

delivery_summary

,Delivery_Time_Min
count,6999.000000
mean,64.708673
min,10.000000
max,119.000000


**Finding:**  
The average delivery time is approximately 64.7 minutes across all orders.

### Customer Rating

In [ ]:
rating_summary = df['Store_Rating'].describe()

rating_summary

,Store_Rating
count,6168.000000
mean,2.980869
std,1.403379
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000


In [ ]:
print("Average Store Rating:", round(df['Store_Rating'].mean(), 2))

Average Store Rating: 2.98


**Findings:**
- The average delivery time is approximately 64.7 minutes across all orders.
- The average store rating is approximately 3.0 out of 5, indicating room for improvement in customer satisfaction.
- Store rating analysis is based on 6,168 non-null ratings due to missing values.

In [ ]:
monthly_analysis = (
    df[df['Month_name'] != 'April']
    .groupby('Month_name')
    .agg(
        Orders=('Transaction_ID', 'nunique'),
        Total_Sales=('Total_Sales', 'sum'),
        Avg_Order_Value=('Total_Sales', 'mean')
    )
    .reindex(['January', 'February', 'March'])
)

monthly_analysis

,Orders,Total_Sales,Avg_Order_Value
Month_name,,,
January,2101,1583952.14,753.903922
February,1985,1477420.38,744.292383
March,2048,1467898.66,716.747393


## 6. Additional Tests / Non-significant Findings

These tests were conducted to examine additional possible relationships in the dataset. The results did not show strong relationships, so they were not considered key business insights.

In [ ]:
delivery_rating_corr = df['Delivery_Time_Min'].corr(df['Store_Rating'])

print("Correlation between Delivery Time and Store Rating:",
      round(delivery_rating_corr, 4))

Correlation between Delivery Time and Store Rating: 0.0094


**Finding:**
The correlation is very close to zero, indicating no meaningful linear relationship between delivery time and store rating in this dataset.

In [ ]:
distance_delivery_corr = df['Delivery_Distance_KM'].corr(df['Delivery_Time_Min'])

print("Correlation between Delivery Distance and Delivery Time:",
      round(distance_delivery_corr, 4))

Correlation between Delivery Distance and Delivery Time: 0.0242


**Finding:**
The correlation is very weak, indicating no meaningful linear relationship between delivery distance and delivery time in the dataset.

In [ ]:
temperature_quantity_corr = df['Temperature_Celsius'].corr(df['Quantity'])

print("Correlation between Temperature and Quantity:",
      round(temperature_quantity_corr, 4))

Correlation between Temperature and Quantity: -0.0037


**Finding:**
The correlation is almost zero, indicating no meaningful linear relationship between temperature and quantity ordered.

In [ ]:
source_delivery = df.groupby('Order_Source').agg(
    Orders=('Transaction_ID', 'nunique'),
    Avg_Delivery_Time=('Delivery_Time_Min', 'mean')
).sort_values('Avg_Delivery_Time', ascending=False)

source_delivery

,Orders,Avg_Delivery_Time
Order_Source,,
ElMenus,1778,65.387514
In-store,1702,64.578731
Talabat,1803,64.553522
Mobile App,1716,64.297203


**Finding:**
Average delivery times are very similar across order sources, with a difference of only about 1 minute between the highest and lowest averages. Therefore, order source does not show a meaningful difference in delivery time.

## 7. Key Insights

### 1. Traditional Category Leads Sales

Traditional generates the highest total sales, mainly driven by its higher order volume, while average quantity per order remains similar across categories.
### 2. Cairo-Nasr City Leads Total Sales

Cairo-Nasr City leads in total sales, driven by the highest order volume, while Cairo-Zamalek has the highest average order value.

### 3. Greater Cairo Has the Highest Sales and AOV

Greater Cairo generates the highest total sales and has the highest average order value, while Delta has a noticeably lower average order value.
### 4. Product Demand Varies Mainly by Order Volume

Qishtouza Nutella has the highest order volume, while Sweet Koshary Mix generates the highest total sales. Product sales are largely aligned with order volume, with relatively similar average quantities across products.


## 8. Recommendations
### 1. Strengthen Traditional Category Performance

Maintain the strong performance of the Traditional category while identifying which products within the category contribute most to its high order volume.

### 2. Investigate Regional Performance Gaps

Review customer demand, pricing, product mix, and marketing activity in Delta to understand why its average order value is lower than Greater Cairo.

### 3. Leverage High-Volume Products

Ensure strong availability of high-order-volume products such as Qishtouza Nutella and evaluate opportunities to increase their average order value through suitable bundles or complementary products.

### 4. Review Discount Strategy

Higher discount levels are associated with lower average sales per order without a consistent increase in quantity. Discount effectiveness should therefore be evaluated using additional profitability or cost data before expanding high discount levels.

### 5. Improve Customer Experience

With an average store rating of approximately 3.0/5, management should investigate the main drivers of customer dissatisfaction, such as service quality, product experience, or other operational factors.